# tutorial 3-progressive experiment building

This notebook is part of the neurodesign-plus 2.0 tutorial audit set.

It teaches the current public workflow:

- `Experiment` stores the requested specification.
- `Design` stores one realized schedule with conceptual-trial and event metadata.
- `Optimisation` searches over designs and the authoritative public selection path is `selected_design(rank)`.

Timing is separated into:

- `event_durations`
- `trial_start_interval`
- `post_event_interval`
- `event_transition_interval`
- `inter_trial_interval`
- optional boundary rests via `rest_every_n_trials` and `rest_interval`


In [1]:
from pathlib import Path
from copy import deepcopy
import json
import os
import warnings

import numpy as np

from neurodesign import Design, Experiment, Optimisation, report

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".tmp_mpl"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore", message='install "ipywidgets" for Jupyter support')
np.set_printoptions(suppress=True, precision=3)

DEFAULT_WEIGHTS = [0.0, 0.5, 0.25, 0.25]

TRIAL_TEMPLATES = [
    {
        "template_id": "standard",
        "trial_type": "standard",
        "events": [
            {"category": "cue_easy", "code": 0, "duration": 0.8},
            {
                "category": "choice_left",
                "code": 2,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
    {
        "template_id": "hint_branch",
        "trial_type": "hint",
        "events": [
            {"category": "cue_hard", "code": 1, "duration": 0.8},
            {"category": "hint", "code": 4, "duration": 0.7},
            {
                "category": "choice_right",
                "code": 3,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
    {
        "template_id": "hold_branch",
        "trial_type": "hold",
        "events": [
            {"category": "cue_easy", "code": 0, "duration": 0.8},
            {"category": "hold", "code": 5, "duration": 0.9},
            {
                "category": "choice_left",
                "code": 2,
                "duration": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            },
            {"category": "feedback", "code": 6, "duration": 1.0},
        ],
    },
]

EVENT_TRANSITION_INTERVAL = {
    "by_event_transition": {
        ("cue_easy", "choice_left"): {"model": "uniform", "min": 0.3, "max": 1.0},
        ("cue_hard", "hint"): {"model": "uniform", "min": 0.3, "max": 0.9},
        ("hint", "choice_right"): {"model": "uniform", "min": 0.2, "max": 0.8},
        ("cue_easy", "hold"): {"model": "uniform", "min": 0.2, "max": 0.8},
        ("hold", "choice_left"): {"model": "uniform", "min": 0.3, "max": 0.9},
        ("choice_left", "feedback"): {"model": "uniform", "min": 0.2, "max": 0.7},
        ("choice_right", "feedback"): {"model": "uniform", "min": 0.2, "max": 0.7},
    }
}

INTER_TRIAL_INTERVAL = {"model": "uniform", "min": 1.0, "mean": 1.45, "max": 1.9}

COMMON_SPEC = {
    "TR": 1.0,
    "P": [0.18, 0.12, 0.16, 0.12, 0.10, 0.10, 0.22],
    "C": [
        [1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1],
        [0, 0, 1, -1, 0, 0, 0],
        [1, -1, 0, 0, 0, 0, 0],
    ],
    "rho": 0.3,
    "n_stimuli": 7,
    "resolution": 0.1,
    "trial_templates": TRIAL_TEMPLATES,
    "trial_start_interval": 0.0,
    "post_event_interval": 0.0,
    "event_transition_interval": EVENT_TRANSITION_INTERVAL,
    "inter_trial_interval": INTER_TRIAL_INTERVAL,
    "rest_interval": 0.0,
    "event_durations": {
        "by_event_category": {
            "cue_easy": 0.8,
            "cue_hard": 0.8,
            "choice_left": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            "choice_right": {"model": "uniform", "min": 1.0, "mean": 1.6, "max": 2.2},
            "hint": 0.7,
            "hold": 0.9,
            "feedback": 1.0,
        }
    },
    "trial_max": 2.2,
}


def score_design(design, weights=DEFAULT_WEIGHTS):
    design.designmatrix().FCalc(weights=weights)
    return design


def build_case10_experiment(
    *,
    seed: int = 12,
    trials: list[dict[str, str]] | None = None,
    trial_template_probabilities: list[float] | None = None,
    n_conceptual_trials: int | None = None,
):
    spec = deepcopy(COMMON_SPEC)
    spec["seed"] = seed
    if trials is not None:
        spec["trials"] = trials
    else:
        spec["trial_template_probabilities"] = (
            [0.4, 0.35, 0.25] if trial_template_probabilities is None else trial_template_probabilities
        )
        spec["n_conceptual_trials"] = 10 if n_conceptual_trials is None else n_conceptual_trials
    return Experiment(**spec)

## Case 10. Canonical integrated workflow
This notebook uses the shared Case 10 specification source, runs optimisation, selects rank 0 through the public API, and exports the selected schedule.

In [2]:
case10_exp = build_case10_experiment(seed=12)
case10_design = score_design(case10_exp.create_design(seed=12))
{
    "n_conceptual_trials": case10_exp.n_conceptual_trials,
    "n_events": len(case10_design.order),
    "trial_template_ids": case10_design.trial_template_ids,
    "trial_starts": case10_design.trial_starts.tolist(),
    "event_index_within_trial": case10_design.event_index_within_trial.tolist(),
    "realized_event_durations": case10_design.realized_event_durations.tolist(),
    "realized_event_transition_intervals": case10_design.realized_event_transition_intervals.tolist(),
    "realized_inter_trial_intervals": case10_design.realized_inter_trial_intervals.tolist(),
    "metrics": {"F": case10_design.F, "Fd": case10_design.Fd, "Ff": case10_design.Ff, "Fc": case10_design.Fc},
}

C:\Users\vguigon\Desktop\Research_directory\Lab_SLD\neurodesign-plus\neurodesign\classes.py:822: UserWarning: the resolution is adjusted to be a multiple of the TR. New resolution: 0.1
  warnings.warn(


{'n_conceptual_trials': 10,
 'n_events': 37,
 'trial_template_ids': ['hint_branch',
  'hold_branch',
  'standard',
  'hold_branch',
  'standard',
  'hint_branch',
  'hold_branch',
  'hold_branch',
  'standard',
  'hint_branch'],
 'trial_starts': [0.0,
  6.9,
  14.700000000000001,
  20.800000000000004,
  28.2,
  33.60000000000001,
  40.30000000000001,
  48.1,
  55.4,
  60.89999999999999],
 'event_index_within_trial': [0,
  1,
  2,
  3,
  0,
  1,
  2,
  3,
  0,
  1,
  2,
  0,
  1,
  2,
  3,
  0,
  1,
  2,
  0,
  1,
  2,
  3,
  0,
  1,
  2,
  3,
  0,
  1,
  2,
  3,
  0,
  1,
  2,
  0,
  1,
  2,
  3],
 'realized_event_durations': [0.8,
  0.7000000000000001,
  1.2000000000000002,
  1.0,
  0.8,
  0.9,
  1.5,
  1.0,
  0.8,
  1.7000000000000002,
  1.0,
  0.8,
  0.9,
  1.3,
  1.0,
  0.8,
  1.6,
  1.0,
  0.8,
  0.7000000000000001,
  1.9000000000000001,
  1.0,
  0.8,
  0.9,
  1.4000000000000001,
  1.0,
  0.8,
  0.9,
  1.3,
  1.0,
  0.8,
  1.4000000000000001,
  1.0,
  0.8,
  0.7000000000000001,
  

In [3]:
case10_opt = Optimisation(
    experiment=build_case10_experiment(seed=12),
    weights=DEFAULT_WEIGHTS,
    preruncycles=1,
    cycles=2,
    seed=101,
    optimisation="simulation",
    G=3,
    I=1,
    outdes=2,
    convergence=1,
    folder=Path("output") / "tutorial_report",
)
case10_opt.optimise()
selected_design = case10_opt.selected_design(0)
{
    "completed_generations": case10_opt.generations_completed,
    "stop_reason": case10_opt.stop_reason,
    "best_score_history": [float(value) for value in case10_opt.optima],
    "selected_rank_0_templates": selected_design.trial_template_ids,
    "selected_rank_0_metrics": {"F": selected_design.F, "Fd": selected_design.Fd, "Ff": selected_design.Ff, "Fc": selected_design.Fc},
    "available_selected_ranks": len(case10_opt.out),
}

{'completed_generations': 2,
 'stop_reason': None,
 'best_score_history': [0.7456523819852432, 0.8417877856439969],
 'selected_rank_0_templates': ['hold_branch',
  'hint_branch',
  'hint_branch',
  'hold_branch',
  'standard',
  'hint_branch',
  'hint_branch',
  'hint_branch',
  'hold_branch',
  'hint_branch'],
 'selected_rank_0_metrics': {'F': 0.8417877856439969,
  'Fd': 1.1416310707608162,
  'Ff': 0.8245014245014245,
  'Fc': 0.2593875765529309},
 'available_selected_ranks': 2}

In [4]:
report_dir = Path("output") / "tutorial_exports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "report.pdf"
report.make_report(case10_opt, report_path)
payload = selected_design.export_payload()
spec = case10_opt.exp.export_specification()
(report_dir / "schedule.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
(report_dir / "specification.json").write_text(json.dumps(spec, indent=2, default=str), encoding="utf-8")
reloaded = json.loads((report_dir / "schedule.json").read_text(encoding="utf-8"))
{
    "report_path": str(report_path),
    "schedule_path": str(report_dir / "schedule.json"),
    "specification_path": str(report_dir / "specification.json"),
    "reloaded_first_event": reloaded["schedule"][0],
    "reloaded_counts": reloaded["counts"],
}

{'report_path': 'output\\tutorial_exports\\report.pdf',
 'schedule_path': 'output\\tutorial_exports\\schedule.json',
 'specification_path': 'output\\tutorial_exports\\specification.json',
 'reloaded_first_event': {'run_event_index': 0,
  'trial_id': 0,
  'trial_index': 0,
  'trial_template_id': 'hold_branch',
  'trial_type_id': 'hold',
  'event_index_within_trial': 0,
  'event_category': 'cue_easy',
  'event_code': 0,
  'trial_start': 0.0,
  'realized_trial_start_interval': 0.0,
  'event_onset': 0.0,
  'realized_event_duration': 0.8,
  'event_offset': 0.8,
  'realized_post_event_interval': 0.0,
  'following_event_transition_interval': 0.6000000000000001,
  'following_inter_trial_interval': None,
  'following_rest_interval': None,
  'trial_end': None,
  'event_duration_rule_id': 'event_durations[cue_easy]',
  'trial_start_rule_id': 'trial_start_interval',
  'post_event_rule_id': 'post_event_interval',
  'event_transition_rule_id': 'event_transition_interval[cue_easy->hold]',
  'inter_tr

In [5]:
run_a = Optimisation(
    experiment=build_case10_experiment(seed=12),
    weights=DEFAULT_WEIGHTS,
    preruncycles=1,
    cycles=1,
    seed=808,
    optimisation="simulation",
    G=2,
    I=1,
    outdes=1,
)
run_b = Optimisation(
    experiment=build_case10_experiment(seed=12),
    weights=DEFAULT_WEIGHTS,
    preruncycles=1,
    cycles=1,
    seed=808,
    optimisation="simulation",
    G=2,
    I=1,
    outdes=1,
)
run_a.optimise()
run_b.optimise()
selected_a = run_a.selected_design(0)
selected_b = run_b.selected_design(0)
payload_a = selected_a.export_payload()
payload_b = selected_b.export_payload()
{
    "same_template_sequence": selected_a.trial_template_ids == selected_b.trial_template_ids,
    "same_realized_durations": payload_a["schedule_arrays"]["realized_event_durations"] == payload_b["schedule_arrays"]["realized_event_durations"],
    "same_scores": payload_a["metrics"] == payload_b["metrics"],
    "same_export_payload": payload_a == payload_b,
}

C:\Users\vguigon\Desktop\Research_directory\Lab_SLD\neurodesign-plus\neurodesign\classes.py:822: UserWarning: the resolution is adjusted to be a multiple of the TR. New resolution: 0.1
  warnings.warn(


{'same_template_sequence': True,
 'same_realized_durations': True,
 'same_scores': True,
 'same_export_payload': True}